对于 Whisper 这种经典的 Encoder-Decoder 架构，这一硬件特性（UMA 统一内存）的物理加成尤为显著：
- Encoder（Prefill 阶段）：负责处理 Log-Mel 频谱特征，属于计算密集型 (Compute-Bound)。其计算复杂度受自注意力机制主导，为 $\mathcal{O}(L_{audio}^2 \cdot d)$（$L_{audio}$ 为序列长度，$d$ 为隐藏层维度）。
- Decoder（Decode 阶段）：采用自回归方式逐字生成文本，属于内存带宽受限型 (Memory-Bandwidth Bound)。生成单个 Token 的耗时 $t_{token}$ 可近似建模为：$$t_{token} \approx \frac{P \times B_{bytes}}{BW}$$

### 加速

- 在 Apple Silicon 上，Encoder 可通过 Core ML 在 ANE 上执行，并且相对纯 CPU 可有明显加速。

### openai-whipser

- parameters
    - `--word_timestamps True`
    - `--condition_on_previous_text False`: prevent hallucination loops
        - 禁止模型在预测当前语音时参考上一段识别出的文本。
        - Whisper 的一个常见痛点——“幻觉死循环”（Hallucination loops）。在遇到长时间静音或背景噪音时，Whisper 经常会像复读机一样疯狂重复上一句话。将此参数设为 False 可以极大地减少这种灾难性错误的发生。
        - 长转录的重复来自 condition-on-previous-text=True 在噪声/片头段落后把错误上下文带进后续窗口。我要用 --condition-on-previous-text False 重新跑
    - --max_line_width 40：字符
    - --max_line_count 1

### whisper.cpp

```
git clone https://github.com/ggml-org/whisper.cpp.git
cd whisper.cpp
```

#### ggml

```
sh ./models/download-ggml-model.sh large-v3-turbo
sh ./models/download-ggml-model.sh large-v3

# build the project
cmake -B build
cmake --build build -j --config Release

# transcribe an audio file
./build/bin/whisper-cli -m models/ggml-large-v3-turbo.bin -f xiesaining.wav -osrt -of ./subs/xiesaining
```
- 音频处理：
    - `ffmpeg -i input.mp3 -ar 16000 -ac 1 -c:a pcm_s16le output.wav`
    - `ffmpeg -i input.mp4 -ar 16000 -ac 1 -c:a pcm_s16le output.wav`
- ggml-large-v3-turbo.bin
    - `whisper_print_timings:    total time = 456610.12 ms`
    - `whisper_print_timings:    total time = 458205.88 ms`
- ggml-large-v3.bin
```
system_info: n_threads = 4 / 18 | WHISPER : COREML = 0 | OPENVINO = 0 | MTL : EMBED_LIBRARY = 1 | CPU : NEON = 1 | ARM_FMA = 1 | FP16_VA = 1 | MATMUL_INT8 = 1 | DOTPROD = 1 | SME = 1 | ACCELERATE = 1 | REPACK = 1 |
```

```
# 为 encoder 生成 Core ML 版本
uv venv .venv-coreml
source .venv-coreml/bin/activate
uv pip install ane_transformers openai-whisper coremltools
./models/generate-coreml-model.sh large-v3-turbo
deactivate

# 编译，打开 Core ML
cmake -B build -DWHISPER_COREML=1
cmake --build build -j --config Release
```

- coreml
    - `system_info: n_threads = 4 / 18 | WHISPER : COREML = 1 | OPENVINO = 0 | MTL : EMBED_LIBRARY = 1 | CPU : NEON = 1 | ARM_FMA = 1 | FP16_VA = 1 | MATMUL_INT8 = 1 | DOTPROD = 1 | SME = 1 | ACCELERATE = 1 | REPACK = 1 |`
    - `whisper_print_timings:    total time = 404225.88 ms`

### mlx-whisper

- mlx-whisper 通常要求 MLX/HuggingFace 格式模型
    - mlx-community/whisper-large-v3-mlx
```shell
  HF_HOME="$HOME/.cache/huggingface" \
  HUGGINGFACE_HUB_CACHE="$HOME/.cache/huggingface/hub" \
  work/.venv/bin/mlx_whisper work/transcript/episode.wav \
    --model mlx-community/whisper-large-v3-mlx \
    --output-dir work/transcript \
    --output-name episode_mlx_largev3 \
    --output-format all \
    --language zh \
    --temperature 0 \
    --condition-on-previous-text True \
    --initial-prompt '张小珺 全球大模型季报 Coding AGI Chatbot Agent Anthropic OpenAI Gemini Meta xAI
  GPT-3 GPT-4 GPT-5 Opus 4.5 Opus 4.6 Claude Sonnet Mythos Spud Harness Engineering ToC Amazon Google 操作
  系统 硅谷御三家 中国御三家 白领通缩 语言即世界 代码即方案' \
    --word-timestamps True \
    --max-line-width 28 \
    --max-line-count 1 \
    --hallucination-silence-threshold 1.0 \
    --verbose False
```